In [1]:
#The LEGB rule - Python looks for names in this order:

In [ ]:
L — Local      (inside current function)
E — Enclosing  (outer function, if nested)
G — Global     (module level)
B — Built-in   (Python's built-ins like len, print)

In [2]:
#Local scope:

In [3]:
def greet():
    message = "Hello"   # local — only exists inside greet
    print(message)

greet()
# print(message)   ← NameError! message doesn't exist here

Hello


In [4]:
#Global scope:

In [5]:
name = "Diya"    # global

def show():
    print(name)  # can READ global

show()    # Diya — works fine

Diya


In [6]:
#The trap - local shadows global:

In [7]:
x = 10

def change():
    x = 99       # creates a NEW local x — doesn't touch global!
    print(x)     # 99

change()
print(x)         # still 10!

99
10


In [8]:
#global keyword - modify global from inside:

In [9]:
count = 0

def increment():
    global count      # declare intent to modify global
    count += 1

increment()
increment()
print(count)   # 2

2


In [10]:
#Enclosing scope (nonlocal):

In [11]:
def outer():
    x = 10

    def inner():
        nonlocal x    # refers to outer's x, not global
        x += 1
        print(x)      # 11

    inner()
    print(x)          # 11 — outer's x was changed

outer()

11
11


In [12]:
#nonlocal vs global:

In [13]:
x = 0

def outer():
    x = 10
    def inner():
        global x     # reaches GLOBAL x=0, not outer's x=10
        x += 1
    inner()
    print(x)         # still 10 — outer's x untouched

outer()
print(x)             # 1 — global x changed

10
1


In [15]:
#Built-in scope - never shadow built-ins:

In [16]:
# NEVER do this:
list = [1, 2, 3]    # shadows built-in list!
print(list([1,2]))  # TypeError — list is now your variable

# If you did it by accident:
del list            # restore access to built-in

TypeError: 'list' object is not callable

In [17]:
#Scope in practice — counter using closure:

In [18]:
def make_counter(start=0):
    count = [start]          # mutable container trick!

    def increment(by=1):
        count[0] += by
        return count[0]

    def reset():
        count[0] = start

    def value():
        return count[0]

    return increment, reset, value

inc, rst, val = make_counter(10)
print(inc())    # 11
print(inc(5))   # 16
print(val())    # 16
rst()
print(val())    # 10

11
16
16
10


In [19]:
#globals() and locals() — inspect scope:

In [20]:
x = 10
y = 20

def show_scope():
    a = 1
    b = 2
    print(locals())    # {'a':1, 'b':2}

show_scope()
print("x" in globals())   # True

{'a': 1, 'b': 2}
True


In [21]:
#examples

In [22]:
#Predict the scope output

In [27]:
print(10)
print(5)

print(10)
print(10)

print(5)

print([1, 2, 3, 4])

print("UnboundLocalError")

10
5
10
10
5
[1, 2, 3, 4]
UnboundLocalError


In [23]:
#Global counter

In [28]:
visit_count = 0

def visit():
    global visit_count
    visit_count += 1
    print(f"Visit #{visit_count}")

def reset_visits():
    global visit_count
    visit_count = 0
    print("Counter reset")

def get_visits():
    return visit_count


visit()
visit()
visit()

print(get_visits())

reset_visits()

visit()

print(get_visits())

Visit #1
Visit #2
Visit #3
3
Counter reset
Visit #1
1


In [24]:
#Nonlocal — closure counter

In [29]:
def make_account(owner, initial_balance=0):
    balance = initial_balance
    transaction_history = []

    def deposit(amount):
        nonlocal balance
        balance += amount
        transaction_history.append(
            f"Deposit: +{amount} | Balance: {balance}"
        )
        return balance

    def withdraw(amount):
        nonlocal balance

        if amount <= balance:
            balance -= amount
            transaction_history.append(
                f"Withdraw: -{amount} | Balance: {balance}"
            )
            return balance
        else:
            transaction_history.append(
                f"Withdraw: FAILED ({amount})"
            )
            return "Insufficient funds"

    def statement():
        print(f"--- {owner}'s Statement ---")

        for transaction in transaction_history:
            print(transaction)

        print(f"Final balance: {balance}")

    return deposit, withdraw, statement


acc = make_account("Diya", 1000)

acc_dep, acc_wit, acc_stmt = acc

print(acc_dep(500))
print(acc_wit(200))
print(acc_wit(2000))

acc_stmt()

1500
1300
Insufficient funds
--- Diya's Statement ---
Deposit: +500 | Balance: 1500
Withdraw: -200 | Balance: 1300
Withdraw: FAILED (2000)
Final balance: 1300


In [25]:
#LEGB tracer

In [30]:
B_val = "built-in(simulated)"
G_val = "global"


def outer(E_val):

    def inner(L_val):
        nonlocal E_val
        global G_val

        print("L:", L_val)
        print("E:", E_val)
        print("G:", G_val)
        print("B:", B_val)

        G_val = "global_modified"
        E_val = "enclosing_modified"

    inner("local")

    print("E after inner:", E_val)


outer("enclosing")

print("G after outer:", G_val)


def scope_demo():
    a = 10
    b = 20

    print("Local vars in scope_demo:", len(locals()))
    print("Global vars count:", len(globals()))


scope_demo()

L: local
E: enclosing
G: global
B: built-in(simulated)
E after inner: enclosing_modified
G after outer: global_modified
Local vars in scope_demo: 2
Global vars count: 80


In [26]:
#Plugin registry using global scope

In [35]:
del list

In [36]:
print(list)

<class 'list'>


In [37]:
my_list = [1, 2, 3]

PLUGINS = {}


def register(name, category="general"):

    def decorator(func):
        PLUGINS[name] = {
            "function": func,
            "category": category,
            "doc": func.__doc__
        }
        return func

    return decorator


def run_plugin(name, *args, **kwargs):
    plugin = PLUGINS.get(name)

    if plugin:
        return plugin["function"](*args, **kwargs)

    return "Plugin not found"


def list_plugins(category=None):

    if category:
        plugins = [
            name for name, info in PLUGINS.items()
            if info["category"] == category
        ]
    else:
        plugins = list(PLUGINS.keys())

    return sorted(plugins)


def plugin_info(name):
    plugin = PLUGINS.get(name)

    if plugin:
        return {
            "name": name,
            "category": plugin["category"],
            "doc": plugin["doc"]
        }

    return None


@register("greet", category="text")
def greet(name):
    """Greets a person by name"""
    return f"Hello {name}!"


@register("double", category="math")
def double(n):
    """Doubles a number"""
    return n * 2


@register("shout", category="text")
def shout(text):
    """Converts text to uppercase"""
    return text.upper() + "!!!"


@register("square", category="math")
def square(n):
    """Squares a number"""
    return n ** 2


print("All plugins:", ", ".join(list_plugins()))
print("Math plugins:", ", ".join(list_plugins("math")))

print(run_plugin("greet", "Diya"))
print(run_plugin("double", 10))
print(run_plugin("shout", "python"))
print(run_plugin("square", 5))

info = plugin_info("greet")

print(
    f"Name: {info['name']} | "
    f"Category: {info['category']} | "
    f"Doc: {info['doc']}"
)

All plugins: double, greet, shout, square
Math plugins: double, square
Hello Diya!
20
PYTHON!!!
25
Name: greet | Category: text | Doc: Greets a person by name
